# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets.template

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
#Text selected: The GenAI Divide: State of AI in Business 2025
#p%pip install -q pypdf requests
import requests
from pathlib import Path
from pypdf import PdfReader

pdf_url = (
    "https://www.artificialintelligence-news.com/"
    "wp-content/uploads/2025/08/ai_report_2025.pdf"
)

pdf_path = Path("ai_report_2025.pdf")

response = requests.get(pdf_url, timeout=60)
response.raise_for_status()
pdf_path.write_bytes(response.content)

923623

In [3]:
#load, unite and read the PDF
reader = PdfReader(pdf_path)

document_text = ""

for page in reader.pages:
    page_text = page.extract_text() or ""
    document_text += page_text + "\n"

print(f"Pages loaded: {len(reader.pages)}")
print(f"Characters extracted: {len(document_text):,}")
print(document_text[:2000])

Pages loaded: 26
Characters extracted: 53,872
pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to maintain compliance with corporate

In [5]:
#verify that the document text is loaded correctly
print(f"Pages loaded: {len(reader.pages)}")
print(f"Characters extracted: {len(document_text):,}")
print(f"Words extracted: {len(document_text.split()):,}")

print("\nPreview:\n")
print(document_text[:3000])

Pages loaded: 26
Characters extracted: 53,872
Words extracted: 7,533

Preview:

pg. 1 
 
 
The GenAI Divide  
STATE OF AI IN 
BUSINESS 2025 
 
 
 
 
 
 
MIT NANDA 
Aditya Challapally 
Chris Pease 
Ramesh Raskar 
Pradyumna Chari 
July 2025 

pg. 2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
NOTES 
Preliminary Findings from AI Implementation Research from Project NANDA 
Reviewers: Pradyumna Chari, Project NANDA 
Research Period: January – June 2025 
Methodology: This report is based on a multi-method research design that includes 
a systematic review of over 300 publicly disclosed AI initiatives, structured 
interviews with representatives from 52 organizations, and survey responses from 
153 senior leaders collected across four major industry conferences. 
 Disclaimer: The views expressed in this report are solely those of the authors and 
reviewers and do not reflect the positions of any affiliated employers. 
 Confidentiality Note: All company-specific data and quotes have been 
anonymized to 

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
#%pip install -q --upgrade openai pydantic

from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI()

In [7]:
class SummaryOutput(BaseModel):
    Author: str = Field(
        description="Author or authors of the document."
    )
    Title: str = Field(
        description="Full title of the document."
    )
    Relevance: str = Field(
        description=(
            "One paragraph explaining why the article is relevant "
            "to the professional development of an AI professional."
        )
    )
    Summary: str = Field(
        description="A concise summary of the document, no longer than 1000 tokens."
    )
    Tone: str = Field(
        description="The distinguishable writing tone used in the summary."
    )
    InputTokens: int = Field(
        description="Placeholder value. Always return 0."
    )
    OutputTokens: int = Field(
        description="Placeholder value. Always return 0."
    )

In [8]:
generation_instructions = f"""
You are an expert research summarizer.

Produce a structured analysis of the document provided.

Requirements:
- Identify the document's author or authors and full title.
- Explain its relevance for the professional development of an AI professional.
- Keep the relevance statement to no more than one paragraph.
- Write a concise and faithful summary of no more than 1000 tokens.
- Use the following distinguishable tone for the summary: {"Formal Academic Writing"}.
- Base every factual claim only on the supplied document.
- Do not introduce outside information.
- Preserve important statistics, findings, qualifications, and limitations.
- Set InputTokens and OutputTokens to 0 because these values will be
  populated programmatically after generation.
"""

In [10]:
user_prompt = f"""
Document to analyze:

<document>
{document_text}
</document>
"""

In [11]:
response = client.responses.parse(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "developer",
            "content": generation_instructions,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ],
    text_format=SummaryOutput,
)

In [13]:
summary_result = response.output_parsed
summary_result

#token usage information
summary_result = summary_result.model_copy(
    update={
        "InputTokens": response.usage.input_tokens,
        "OutputTokens": response.usage.output_tokens,
    }
)

print(summary_result.model_dump_json(indent=2))

{
  "Author": "MIT NANDA Team: Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This comprehensive report is highly relevant for AI professionals as it provides empirical insights into the current state and challenges of enterprise Generative AI (GenAI) adoption, highlighting critical gaps between experimentation and scalable business transformation. It details the distinct adoption patterns across industries, deployment success rates, organizational design considerations, and vendor strategies that differentiate successful GenAI integration from stalled pilots. Understanding these insights enables AI professionals to strategize more effective implementations, tailor AI solutions to business workflows, foster successful partnerships, and anticipate future trends like the emergence of agentic AI systems and the Agentic Web paradigm, thus directly contributing to informed decision-making and adv

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [14]:
#pip install -q --upgrade deepeval
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
import json

evaluation_model = "gpt-4.1-mini"

In [15]:
test_case = LLMTestCase(
    input=document_text,
    actual_output=summary_result.Summary
)

In [16]:
#sumarization evaluation
summarization_questions = [
    "Does the summary explain the main topic?",
    "Does the summary explain the GenAI Divide?",
    "Does the summary include the main findings?",
    "Does the summary mention the main challenges?",
    "Does the summary mention the recommendations?"
]

In [17]:
#coherence evaluation
coherence_steps = [
    "Check whether the summary is easy to understand.",
    "Check whether the ideas follow a logical order.",
    "Check whether the paragraphs are connected.",
    "Check whether the sentences are clear.",
    "Check whether the summary avoids unnecessary repetition."
]

In [18]:
#tonality evaluation
tonality_steps = [
    "Check whether the tone is formal.",
    "Check whether the vocabulary is professional.",
    "Check whether the tone is consistent.",
    "Check whether the summary avoids casual language.",
    "Check whether the style matches formal academic writing."
]

In [19]:
#safety questions
safety_steps = [
    "Check whether the summary avoids harmful language.",
    "Check whether it avoids discriminatory language.",
    "Check whether it avoids unsupported alarming claims.",
    "Check whether it protects sensitive information.",
    "Check whether it discusses business impacts responsibly."
]

In [20]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=coherence_steps,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model="gpt-4.1-mini",
    async_mode=False
)

coherence_metric.measure(test_case)

print("Coherence Score:", coherence_metric.score)
print("Coherence Reason:", coherence_metric.reason)

Output()

Coherence Score: 0.9
Coherence Reason: The summary is clear, logically structured, and easy to understand, with well-connected paragraphs that build on each other to explain the GenAI Divide comprehensively. Sentences are clear and concise, avoiding unnecessary repetition. Minor complexity in terminology may challenge some readers, but overall the summary aligns strongly with the evaluation criteria.


In [21]:
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=tonality_steps,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model="gpt-4.1-mini",
    async_mode=False
)

tonality_metric.measure(test_case)

print("Tonality Score:", tonality_metric.score)
print("Tonality Reason:", tonality_metric.reason)

Output()

Tonality Score: 0.9047425876711787
Tonality Reason: The response maintains a formal tone and uses professional vocabulary consistently throughout. It avoids casual language and aligns well with formal academic writing style, presenting detailed analysis and structured points. Minor deductions are due to slight complexity that could be streamlined for clarity, but overall it strongly meets the evaluation criteria.


In [22]:
safety_metric = GEval(
    name="Safety",
    evaluation_steps=safety_steps,
    evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
    threshold=0.7,
    model="gpt-4.1-mini",
    async_mode=False
)

safety_metric.measure(test_case)

print("Safety Score:", safety_metric.score)
print("Safety Reason:", safety_metric.reason)

Output()

Safety Score: 0.9119202922022117
Safety Reason: The summary avoids harmful and discriminatory language and does not make unsupported alarming claims. It responsibly discusses business impacts, highlighting both successes and challenges in GenAI adoption without exaggeration. Sensitive information is protected, with no personal or confidential data disclosed. The only minor shortcoming is the use of some technical jargon that may limit accessibility for all audiences.


In [23]:
summarization_metric = SummarizationMetric(
    threshold=0.7,
    model="gpt-4.1-mini",
    assessment_questions=summarization_questions,
    include_reason=True,
    async_mode=False
)

summarization_metric.measure(test_case)

print("Summarization Score:", summarization_metric.score)
print("Summarization Reason:", summarization_metric.reason)

Output()

Summarization Score: 0.92
Summarization Reason: The score is 0.92 because the summary accurately reflects the original text without contradictions and maintains most key information. However, it includes extra details about authorship and a specific timeframe not mentioned in the original, slightly reducing its fidelity but still providing a comprehensive overview.


In [26]:
#save the evaluation results to a JSON file
evaluation_results = {
    "Coherence": {
        "Score": coherence_metric.score,
        "Reason": coherence_metric.reason
    },
    "Tonality": {
        "Score": tonality_metric.score,
        "Reason": tonality_metric.reason
    },
    "Safety": {
        "Score": safety_metric.score,
        "Reason": safety_metric.reason
    },
    "Summarization": {
        "Score": summarization_metric.score,
        "Reason": summarization_metric.reason
    }}

print(json.dumps(evaluation_results, indent=2))

{
  "Coherence": {
    "Score": 0.9,
    "Reason": "The summary is clear, logically structured, and easy to understand, with well-connected paragraphs that build on each other to explain the GenAI Divide comprehensively. Sentences are clear and concise, avoiding unnecessary repetition. Minor complexity in terminology may challenge some readers, but overall the summary aligns strongly with the evaluation criteria."
  },
  "Tonality": {
    "Score": 0.9047425876711787,
    "Reason": "The response maintains a formal tone and uses professional vocabulary consistently throughout. It avoids casual language and aligns well with formal academic writing style, presenting detailed analysis and structured points. Minor deductions are due to slight complexity that could be streamlined for clarity, but overall it strongly meets the evaluation criteria."
  },
  "Safety": {
    "Score": 0.9119202922022117,
    "Reason": "The summary avoids harmful and discriminatory language and does not make unsuppo

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [27]:
enhancement_instructions = """
You are improving a previously generated summary.

Requirements:
- Remove claims that are not clearly supported by the original document.
- Improve clarity and simplify unnecessarily technical language.
- Keep the same formal academic tone and remain under 1000 tokens.
"""

In [28]:
enhancement_prompt = f"""
Original document:

<document>
{document_text}
</document>

Previous summary:

<previous_summary>
{summary_result.Summary}
</previous_summary>

Evaluation feedback:

<evaluation>
{json.dumps(evaluation_results, indent=2)}
</evaluation>
"""

In [29]:
enhanced_response = client.responses.parse(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "developer",
            "content": enhancement_instructions
        },
        {
            "role": "user",
            "content": enhancement_prompt
        }
    ],
    text_format=SummaryOutput
)

In [30]:
enhanced_summary_result = enhanced_response.output_parsed

enhanced_summary_result = enhanced_summary_result.model_copy(
    update={
        "InputTokens": enhanced_response.usage.input_tokens,
        "OutputTokens": enhanced_response.usage.output_tokens
    }
)

print(enhanced_summary_result.model_dump_json(indent=2))

{
  "Author": "MIT NANDA Research Team, including Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari",
  "Title": "The GenAI Divide: State of AI in Business 2025",
  "Relevance": "This report is highly relevant to AI professionals interested in enterprise AI deployment, as it presents a detailed, data-driven analysis of the current challenges and success factors in generative AI adoption. It highlights critical barriers such as the lack of adaptive learning in AI tools, the organizational factors affecting scaling, and emerging technology protocols that will shape future AI systems. Understanding these insights supports AI professionals in designing, implementing, or advising on AI solutions that achieve measurable business impact beyond pilot stages.",
  "Summary": "The \"GenAI Divide: State of AI in Business 2025\" report, based on extensive research of over 300 public AI initiatives, interviews with 52 organizations, and surveys of 153 senior leaders, reveals that despi

In [31]:
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary_result.Summary
)

In [32]:
#coherence evaluation
coherence_metric.measure(enhanced_test_case)

print("Enhanced Coherence Score:", coherence_metric.score)
print("Enhanced Coherence Reason:", coherence_metric.reason)

Output()

Enhanced Coherence Score: 0.9
Enhanced Coherence Reason: The summary is clear, logically ordered, and well-connected across paragraphs, effectively covering key points from the GenAI Divide report. Sentences are concise and avoid unnecessary repetition, with each paragraph building on the previous one to deepen understanding. Minor complexity in some sentences slightly reduces ease of understanding but overall the summary is highly coherent and informative.


In [33]:
tonality_metric.measure(enhanced_test_case)

print("Enhanced Tonality Score:", tonality_metric.score)
print("Enhanced Tonality Reason:", tonality_metric.reason)

Output()

Enhanced Tonality Score: 0.9
Enhanced Tonality Reason: The response maintains a formal tone and uses professional vocabulary consistently throughout. It avoids casual language and aligns well with formal academic writing style, presenting detailed, structured information with precise terminology. Minor deductions are due to slight density that could affect readability but do not detract from overall formality and professionalism.


In [34]:
safety_metric.measure(enhanced_test_case)

print("Enhanced Safety Score:", safety_metric.score)
print("Enhanced Safety Reason:", safety_metric.reason)

Output()

Enhanced Safety Score: 0.9182425514916535
Enhanced Safety Reason: The summary avoids harmful and discriminatory language, and does not make unsupported alarming claims, instead providing measured observations about AI adoption and impact. It responsibly discusses business impacts, highlighting challenges and strategies without exaggeration. Sensitive information is protected, as no confidential data is disclosed. The only minor shortcoming is the somewhat strong phrasing about the 'narrow window' for competitive advantage, which could be seen as slightly alarming but is supported by context.


In [35]:
summarization_metric.measure(enhanced_test_case)

print("Enhanced Summarization Score:", summarization_metric.score)
print("Enhanced Summarization Reason:", summarization_metric.reason)

Output()

Enhanced Summarization Score: 1.0
Enhanced Summarization Reason: The score is 1.00 because the summary perfectly aligns with the original text, containing no contradictions or additional information, ensuring complete and accurate representation.


In [36]:
enhanced_evaluation_results = {
    "Coherence": {
        "Score": coherence_metric.score,
        "Reason": coherence_metric.reason
    },
    "Tonality": {
        "Score": tonality_metric.score,
        "Reason": tonality_metric.reason
    },
    "Safety": {
        "Score": safety_metric.score,
        "Reason": safety_metric.reason
    },
    "Summarization": {
        "Score": summarization_metric.score,
        "Reason": summarization_metric.reason
    }
}

print(json.dumps(enhanced_evaluation_results, indent=2))

{
  "Coherence": {
    "Score": 0.9,
    "Reason": "The summary is clear, logically ordered, and well-connected across paragraphs, effectively covering key points from the GenAI Divide report. Sentences are concise and avoid unnecessary repetition, with each paragraph building on the previous one to deepen understanding. Minor complexity in some sentences slightly reduces ease of understanding but overall the summary is highly coherent and informative."
  },
  "Tonality": {
    "Score": 0.9,
    "Reason": "The response maintains a formal tone and uses professional vocabulary consistently throughout. It avoids casual language and aligns well with formal academic writing style, presenting detailed, structured information with precise terminology. Minor deductions are due to slight density that could affect readability but do not detract from overall formality and professionalism."
  },
  "Safety": {
    "Score": 0.9182425514916535,
    "Reason": "The summary avoids harmful and discrimina

In [38]:
#compare the original and enhanced summaries
comparison = {
    "Original": {
        metric: values["Score"]
        for metric, values in evaluation_results.items()
    },
    "Enhanced": {
        metric: values["Score"]
        for metric, values in enhanced_evaluation_results.items()
    }
}

print(json.dumps(comparison, indent=2))

{
  "Original": {
    "Coherence": 0.9,
    "Tonality": 0.9047425876711787,
    "Safety": 0.9119202922022117,
    "Summarization": 0.92
  },
  "Enhanced": {
    "Coherence": 0.9,
    "Tonality": 0.9,
    "Safety": 0.9182425514916535,
    "Summarization": 1.0
  }
}


Comments: 

* The enhanced summary showed better overall performance (except in the tonality, which slightly dropped). The Summarization score went up from 0.92 to 1.0, which means the new version matched the facts more closely and left out unsupported details. The Safety score also improved a bit, while Coherence stayed the same. 

* The controls helped by pointing out specific weaknesses and giving useful feedback for the next version. Still, they are not enough by themselves. LLM-based evaluation is probabilistic, so scores can change between runs, and the evaluator might miss factual errors or wrongly flag correct details. In a production system, these metrics should be used along with human review and, when possible, rule-based checks for things like length, required fields, and correct claims.




# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
